## RQ3: Predicting flight delays

### Setup and imports

In [1]:
from pathlib import Path
import sys
import json


import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
from xgboost import XGBClassifier

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.config import CLEAN_PARQUET, TABLES_DIR
from src.data_loading import load_flights

In [2]:
FLIGHTS_SRC = "flights_2024_clean.parquet"

BASELINE_COLS = [
    ("route_past_delay_rate", "derived from training data separately for each split"),
]
FLIGHT_COLS = [  # standard features from the original flight dataframe
    ("month", FLIGHTS_SRC),
    ("day_of_week", FLIGHTS_SRC),
    ("sched_hour", f"{FLIGHTS_SRC}: crs_dep_time // 100"),
    ("op_unique_carrier", FLIGHTS_SRC),
    ("distance", FLIGHTS_SRC),
]
RQ1_COLS = [
    ("route_cluster", "rq1_route_clusters_jan_jun.csv (joined on origin, dest, op_unique_carrier)"),
    ("cluster_past_delay_rate", "rq1_cluster_delay_rates_jan_jun.csv: historical_delay_rate (joined on route_cluster)"),
]
RQ2_COLS = [
    ("origin_community", "03_airports.csv: louvain (joined on origin)"),
    ("dest_community", "03_airports.csv: louvain (joined on dest)"),
    ("crosses_community", "derived: origin_community != dest_community"),
    ("origin_betweenness", "03_airports.csv: betweenness (joined on origin)"),
    ("dest_betweenness", "03_airports.csv: betweenness (joined on dest)"),
]
OTHER_COLS = [  # not features: route key for the baseline, and the target
    ("origin", FLIGHTS_SRC),
    ("dest", FLIGHTS_SRC),
    ("late_arrival", f"target, {FLIGHTS_SRC}: arr_delay >= 15"),
]

ROUTE_KEYS = ["origin", "dest", "op_unique_carrier"]

BASELINE_FEATURES = [column for column, _ in BASELINE_COLS]
FLIGHT_FEATURES = [column for column, _ in FLIGHT_COLS]
RQ1_FEATURES = [column for column, _ in RQ1_COLS]
RQ2_FEATURES = [column for column, _ in RQ2_COLS]


In [3]:
# Delay columns are loaded only to build the target, never as features.
flights = load_flights(
    CLEAN_PARQUET,
    columns=["month", "day_of_week", "op_unique_carrier", "origin", "dest",
             "crs_dep_time", "distance", "status", "arr_delay"],
)

# Cancelled and diverted flights have no arrival delay.
flights = flights[flights["status"] == "completed"].dropna(subset=["arr_delay"]).reset_index(drop=True)

# Late = arrives 15+ minutes late (official FAA definition).
flights["late_arrival"] = (flights["arr_delay"] >= 15).astype(int)
flights["sched_hour"] = ((flights["crs_dep_time"] // 100) % 24).astype(int)
flights = flights.drop(columns=["status", "arr_delay", "crs_dep_time"])

# RQ1 output: route + airline -> cluster, and the cluster's past delay rate.
route_lookup = pd.read_csv(TABLES_DIR / "rq1_route_clusters_jan_jun.csv")
route_rates = pd.read_csv(TABLES_DIR / "rq1_cluster_delay_rates_jan_jun.csv")[
    ["route_cluster", "historical_delay_rate"]
].rename(columns={"historical_delay_rate": "cluster_past_delay_rate"})

flights = flights.merge(route_lookup, on=ROUTE_KEYS, how="left", validate="many_to_one")
flights = flights.merge(route_rates, on="route_cluster", how="left", validate="many_to_one")

# Routes not seen in Jan-Jun have no cluster.
flights["route_cluster"] = flights["route_cluster"].astype("Int64").astype("string").fillna("unknown")

# RQ2 output: community and betweenness of the origin and of the destination airport.
airports = pd.read_csv(TABLES_DIR / "03_airports.csv")[["airport", "louvain", "betweenness"]]
for side in ["origin", "dest"]:
    side_features = airports.rename(columns={
        "airport": side,
        "louvain": f"{side}_community",
        "betweenness": f"{side}_betweenness",
    })
    flights = flights.merge(side_features, on=side, how="left", validate="many_to_one")
    flights[f"{side}_community"] = flights[f"{side}_community"].astype("Int64").astype("string").fillna("unknown")

flights["crosses_community"] = (flights["origin_community"] != flights["dest_community"]).astype(int)

rq3_df = flights[["origin", "dest"] + FLIGHT_FEATURES + RQ1_FEATURES + RQ2_FEATURES + ["late_arrival"]].copy()

print(f"RQ3 modeling dataframe rows: {len(rq3_df):,}")
print(f"Target rate: {rq3_df['late_arrival'].mean():.3%}")
print(f"Unknown route cluster: {(rq3_df['route_cluster'] == 'unknown').mean():.2%}")
print(f"Unknown origin / dest community: {(rq3_df['origin_community'] == 'unknown').mean():.2%} / {(rq3_df['dest_community'] == 'unknown').mean():.2%}")


feature_groups = [
    ("Baseline", BASELINE_COLS),
    ("standard_flight", FLIGHT_COLS),
    ("RQ1", RQ1_COLS),
    ("RQ2", RQ2_COLS),
    ("Other (not features)", OTHER_COLS),
]
for name, cols in feature_groups:
    print(f"{name}")
    for col, src in cols:
        print(f"    {col:<24} <- {src}")

# Dataset columns are accounted for; route keys and the target are not model features.
listed = [c for c, _ in FLIGHT_COLS + RQ1_COLS + RQ2_COLS + OTHER_COLS]
assert sorted(listed) == sorted(rq3_df.columns)


RQ3 modeling dataframe rows: 6,965,246
Target rate: 20.817%
Unknown route cluster: 1.83%
Unknown origin / dest community: 0.00% / 0.00%
Baseline
    route_past_delay_rate    <- derived from training data separately for each split
standard_flight
    month                    <- flights_2024_clean.parquet
    day_of_week              <- flights_2024_clean.parquet
    sched_hour               <- flights_2024_clean.parquet: crs_dep_time // 100
    op_unique_carrier        <- flights_2024_clean.parquet
    distance                 <- flights_2024_clean.parquet
RQ1
    route_cluster            <- rq1_route_clusters_jan_jun.csv (joined on origin, dest, op_unique_carrier)
    cluster_past_delay_rate  <- rq1_cluster_delay_rates_jan_jun.csv: historical_delay_rate (joined on route_cluster)
RQ2
    origin_community         <- 03_airports.csv: louvain (joined on origin)
    dest_community           <- 03_airports.csv: louvain (joined on dest)
    crosses_community        <- derived: origin_communit

### Feature groups

In [4]:
for name, features in [
    ("Baseline", BASELINE_FEATURES),
    ("standard_flight", FLIGHT_FEATURES),
    ("RQ1", RQ1_FEATURES),
    ("RQ2", RQ2_FEATURES),
]:
    print(f"{name}: {features}")

Baseline: ['route_past_delay_rate']
standard_flight: ['month', 'day_of_week', 'sched_hour', 'op_unique_carrier', 'distance']
RQ1: ['route_cluster', 'cluster_past_delay_rate']
RQ2: ['origin_community', 'dest_community', 'crosses_community', 'origin_betweenness', 'dest_betweenness']


### Feature configurations
These eight configurations are preserved exactly. The names `Baseline + RQ1`,
`Baseline + RQ2`, and `Baseline + RQ1 + RQ2` also include `standard_flight`;
inspect the table below for the exact inputs.


In [5]:
# Helper function to combine features without duplicates
def combine_features(*feature_lists):
    return list(dict.fromkeys(
        col
        for features in feature_lists
        for col in features
    ))


FEATURE_CONFIGS = {
    "Baseline + RQ1 + RQ2": combine_features(
        BASELINE_FEATURES,
        FLIGHT_FEATURES,
        RQ1_FEATURES,
        RQ2_FEATURES
    )
}

In [7]:
feature_config_table = pd.DataFrame([
    {"Feature Config": name, "Number of Features": len(features),
     "Features": ", ".join(features)}
    for name, features in FEATURE_CONFIGS.items()
])
display(feature_config_table)


,Feature Config,Number of Features,Features
0,Baseline + RQ1 + RQ2,13,"route_past_delay_rate, month, day_of_week, sch..."


### Expanding-window feature selection
Each configuration is fitted from scratch twice:

| Fold | Training months | Validation month |
| --- | --- | --- |
| July | 1–6 | 7 |
| August | 1–7 | 8 |


In [8]:
VALIDATION_FOLDS = (
    {"fold": "July", "train_months": tuple(range(1, 7)), "validation_month": 7},
)
MODEL_TYPE = "EBM"
SELECTION_METRIC = "PR-AUC"
RANDOM_STATE = 42
# None fits every training row. Set e.g. 200_000 for a quicker comparison;
# all configurations then receive the same sample within each fold.
SAMPLE_SIZE = 200_000
THRESHOLD = 0.5


def make_model(model_type=MODEL_TYPE, random_state=RANDOM_STATE):
    if model_type == "EBM":
        return ExplainableBoostingClassifier(random_state=random_state, interactions=0)
    if model_type == "XGBoost":
        return XGBClassifier(
            n_estimators=100, max_depth=6, enable_categorical=True,
            tree_method="hist", random_state=random_state,
            eval_metric="logloss",
        )
    raise ValueError("model_type must be 'EBM' or 'XGBoost'")


def add_historical_rate(
    train, val, keys, output_col, smoothing=20, initial_prior=0.5
):
    """Training: earlier months only. Validation: full training history."""
    train = train.copy()
    val = val.copy()

    if train.empty:
        raise ValueError("Training data must not be empty")
    if train["month"].isna().any() or val["month"].isna().any():
        raise ValueError("Month must not be missing")
    if not val.empty and train["month"].max() >= val["month"].min():
        raise ValueError("Validation must follow the training months")
    if smoothing < 0:
        raise ValueError("smoothing must be non-negative")

    def encode(history, rows):
        if history.empty:
            # Fixed prior: do not estimate it from future outcomes.
            return pd.Series(initial_prior, index=rows.index, dtype=float)

        prior = history["late_arrival"].mean()

        # Unknown cluster membership is not a meaningful shared cluster.
        if keys == ["route_cluster"]:
            history = history.loc[
                history["route_cluster"].notna()
                & history["route_cluster"].ne("unknown")
            ]

        stats = (
            history.groupby(keys, observed=True, dropna=False)["late_arrival"]
            .agg(["sum", "count"])
            .reset_index()
        )
        stats["_rate"] = (
            stats["sum"] + smoothing * prior
        ) / (stats["count"] + smoothing)

        # Left merge preserves row order; restore the original index.
        encoded = rows[keys].merge(
            stats[keys + ["_rate"]],
            on=keys,
            how="left",
            validate="many_to_one",
            sort=False,
        )["_rate"].fillna(prior)

        return pd.Series(encoded.to_numpy(), index=rows.index)

    train[output_col] = float(initial_prior)

    for month in sorted(train["month"].unique()):
        mask = train["month"].eq(month)
        history = train.loc[train["month"] < month]
        train.loc[mask, output_col] = encode(
            history, train.loc[mask]
        )

    val[output_col] = encode(train, val)
    return train, val


def prepare_fold(df, fold, sample_size=None, random_state=42):
    train = df[df["month"].isin(fold["train_months"])].copy()
    val = df[df["month"] == fold["validation_month"]].copy()
    if train.empty or val.empty:
        raise ValueError(f"Empty training or validation data for {fold['fold']}")
    missing_months = set(fold["train_months"]) - set(train["month"].unique())
    if missing_months:
        raise ValueError(f"Missing training months: {sorted(missing_months)}")
    if train["late_arrival"].nunique() != 2 or val["late_arrival"].nunique() != 2:
        raise ValueError("Both target classes are required in training and validation")
    train, val = add_historical_rate(train, val, ROUTE_KEYS, "route_past_delay_rate")
    train, val = add_historical_rate(train, val, ["route_cluster"], "cluster_past_delay_rate")
    n_available = len(train)
    if sample_size is not None:
        if sample_size < 2:
            raise ValueError("sample_size must be at least 2 or None")
        if len(train) > sample_size:
            train = train.sample(n=sample_size, random_state=random_state)
    if train["late_arrival"].nunique() != 2:
        raise ValueError("Training sample must contain both target classes")
    # Learn categorical vocabularies from training only. Unseen validation
    # categories become missing values, supported by both estimators.
    all_features = list(dict.fromkeys(f for fs in FEATURE_CONFIGS.values() for f in fs))
    for col in all_features:
        if (pd.api.types.is_object_dtype(train[col])
                or isinstance(train[col].dtype, (pd.StringDtype, pd.CategoricalDtype))):
            categories = pd.Index(train[col].dropna().unique())
            dtype = pd.CategoricalDtype(categories=categories)
            train[col] = train[col].astype(dtype)
            val[col] = val[col].astype(dtype)
    return train, val, n_available


def evaluate_model(model, X, y, threshold=0.5):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y, probs),
        "PR-AUC": average_precision_score(y, probs),
        "Log Loss": log_loss(y, probs, labels=[0, 1]),
        "Brier Score": brier_score_loss(y, probs),
        "Accuracy": accuracy_score(y, preds),
        "Precision": precision_score(y, preds, zero_division=0),
        "Recall": recall_score(y, preds, zero_division=0),
        "F1": f1_score(y, preds, zero_division=0),
    }


def run_feature_selection(df, model_type=MODEL_TYPE, sample_size=SAMPLE_SIZE):
    rows = []
    for fold in VALIDATION_FOLDS:
        train, val, n_available = prepare_fold(df, fold, sample_size, RANDOM_STATE)
        for name, features in FEATURE_CONFIGS.items():
            print(f"{model_type} | {name} | train 1–{max(fold['train_months'])} → validate {fold['validation_month']}")
            model = make_model(model_type, RANDOM_STATE)
            start = time.perf_counter()
            model.fit(train[features], train["late_arrival"])
            duration = time.perf_counter() - start
            for split, data in (("Train", train), ("Validation", val)):
                rows.append({
                    "Estimator": model_type, "Feature Config": name,
                    "Number of Features": len(features), "Fold": fold["fold"],
                    "Train Period": f"1–{max(fold['train_months'])}",
                    "Validation Month": fold["validation_month"], "Split": split,
                    "Available Train Samples": n_available, "Train Samples": len(train),
                    "Eval Samples": len(data), "Training Seconds": duration,
                    **evaluate_model(model, data[features], data["late_arrival"], THRESHOLD),
                })
            # Only results are retained, avoiding 16 large fitted models in memory.
    results = pd.DataFrame(rows)
    assert len(results) == len(FEATURE_CONFIGS) * len(VALIDATION_FOLDS) * 2
    return results


In [9]:
# WARNING : This takes a shit ton of time
results_df = run_feature_selection(rq3_df, model_type=MODEL_TYPE, sample_size=SAMPLE_SIZE)
validation_results = results_df[results_df["Split"] == "Validation"].copy()
display(validation_results.round(4))


C:\Users\huste\AppData\Local\Temp\ipykernel_336296\721301903.py:117: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  val[col] = val[col].astype(dtype)
C:\Users\huste\AppData\Local\Temp\ipykernel_336296\721301903.py:117: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  val[col] = val[col].astype(dtype)


EBM | Baseline + RQ1 + RQ2 | train 1–6 → validate 7


,Estimator,Feature Config,Number of Features,Fold,Train Period,Validation Month,Split,Available Train Samples,Train Samples,Eval Samples,Training Seconds,ROC-AUC,PR-AUC,Log Loss,Brier Score,Accuracy,Precision,Recall,F1
1,EBM,Baseline + RQ1 + RQ2,13,July,1–6,7,Validation,3403472,200000,613938,22.8638,0.6717,0.4356,0.5762,0.1957,0.7046,0.5207,0.0344,0.0646
